# Parquet Basics — 06: Schema Evolution

## What you will learn
Data schemas change over time — new columns are added, old ones removed.
Parquet handles many schema changes gracefully through **schema evolution**.

In this notebook:
1. Adding columns — backward compatible, safe
2. Dropping columns — forward compatible
3. Renaming columns — requires care
4. Type promotion — safe vs unsafe changes
5. `mergeSchema` — handling multiple file versions
6. Schema registry pattern — tracking schema history


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/parquet_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('parquet-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

random.seed(42)
N = 200_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("discount",    DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("is_returned", BooleanType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2023, 1, 1)
rows = []
for i in range(N):
    qty  = random.randint(1, 20)
    price = round(random.uniform(5, 2000), 2)
    disc  = round(random.uniform(0, 0.4), 3)
    rev   = round(qty * price * (1 - disc), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 365))
    rows.append((i+1, random.randint(1,50000),
                 f"Product_{random.randint(1,500)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, disc, rev,
                 random.choice(STATUSES), random.random() < 0.05, d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset: {N:,} rows | {len(schema)} columns")
df.printSchema()

## Step 1 — Adding Columns (Backward Compatible)

Adding a column with a default value is the safest schema change.
Old files simply return null for the new column.


In [ ]:
BASE_PATH = f"{DATA_DIR}/schema_evolution"

# V1: original schema
v1_df = df.select("order_id","customer_id","product","category","revenue","order_date")
v1_path = f"{BASE_PATH}/v1"
v1_df.write.mode("overwrite").parquet(v1_path)
print(f"V1 schema: {v1_df.columns}")
print(f"V1 rows  : {v1_df.count():,}")

# V2: added 'discount' and 'country' columns
v2_df = df.select("order_id","customer_id","product","category",
                   "revenue","order_date","discount","country")
v2_path = f"{BASE_PATH}/v2"
v2_df.write.mode("overwrite").parquet(v2_path)
print(f"\nV2 schema: {v2_df.columns}")
print(f"V2 rows  : {v2_df.count():,}")

# Read V1 files with V2 schema (new cols = null)
v2_schema = spark.read.parquet(v2_path).schema
df_v1_as_v2 = spark.read.schema(v2_schema).parquet(v1_path)
print(f"\nV1 data read with V2 schema (new columns = null):")
df_v1_as_v2.select("order_id","revenue","discount","country").show(5)
print("discount and country are null for V1 rows — safe!")

In [ ]:
# Read both V1 and V2 together using mergeSchema
combined = spark.read.option("mergeSchema", True).parquet(v1_path, v2_path)
print(f"Combined schema (V1 + V2 union): {combined.columns}")
print(f"Total rows: {combined.count():,}")
print()
combined.select("order_id","revenue","discount","country").show(8)
print("V1 rows show null for discount/country — V2 rows show real values")

## Step 2 — Removing Columns

When you stop writing a column, old files still have it.
New files won't have it. Reading with mergeSchema handles this.


In [ ]:
# V3: removed 'discount', added 'loyalty_points'
v3_df = df.select("order_id","customer_id","product","category",
                   "revenue","order_date","country") \
           .withColumn("loyalty_points", (F.col("revenue") * 10).cast("int"))

v3_path = f"{BASE_PATH}/v3"
v3_df.write.mode("overwrite").parquet(v3_path)
print(f"V3 schema: {v3_df.columns}")
print("(discount removed, loyalty_points added)")
print()

# Read all versions together
all_versions = spark.read.option("mergeSchema", True) \
                         .parquet(v1_path, v2_path, v3_path)

print(f"All versions schema: {all_versions.columns}")
print(f"Total rows: {all_versions.count():,}")
print()
all_versions.select("order_id","discount","country","loyalty_points").show(8)
print("V1 rows: null for discount, country, loyalty_points")
print("V2 rows: has discount/country, null for loyalty_points")
print("V3 rows: null for discount, has country and loyalty_points")

## Step 3 — Type Promotion: Safe vs Unsafe

Some type changes are safe (widening), others corrupt data.

| Change | Safe? | Notes |
|---|---|---|
| `INT` → `LONG` | ✅ Safe | Values always fit |
| `FLOAT` → `DOUBLE` | ✅ Safe | More precision |
| `INT` → `DOUBLE` | ✅ Safe | Widening |
| `LONG` → `INT` | ❌ Unsafe | Overflow possible |
| `DOUBLE` → `FLOAT` | ❌ Unsafe | Precision loss |
| `STRING` → `INT` | ❌ Unsafe | Parse errors |
| `INT` → `STRING` | ⚠️ Works | But loses numeric semantics |


In [ ]:
# Safe: INT → LONG
int_df = df.select("order_id","quantity").withColumn("quantity", F.col("quantity").cast("int"))
int_path = f"{BASE_PATH}/type_int"
int_df.write.mode("overwrite").parquet(int_path)

long_df = df.select("order_id","quantity").withColumn("quantity", F.col("quantity").cast("long"))
long_path = f"{BASE_PATH}/type_long"
long_df.write.mode("overwrite").parquet(long_path)

# Read INT file with LONG schema (safe widening)
long_schema = spark.read.parquet(long_path).schema
result_safe = spark.read.schema(long_schema).parquet(int_path)
print("INT → LONG (safe widening):")
result_safe.printSchema()
result_safe.show(3)
print("✅ All values preserved correctly")

# Unsafe: LONG → INT (overflow risk)
print("\nLONG → INT (unsafe narrowing) — Spark reads but may truncate:")
int_schema = spark.read.parquet(int_path).schema
result_unsafe = spark.read.schema(int_schema).parquet(long_path)
result_unsafe.show(3)
print("⚠️  Values may be silently truncated — avoid type narrowing")

## Step 4 — Renaming Columns

Parquet doesn't store column aliases — renaming breaks backward compatibility.
The safe approach: add new column, migrate, then drop old.


In [ ]:
# Problem: simply renaming breaks reading old files
old_df = df.select("order_id", "customer_id", "revenue")
new_df = df.select("order_id",
                   F.col("customer_id").alias("cust_id"),   # renamed!
                   "revenue")

old_path = f"{BASE_PATH}/rename_old"
new_path = f"{BASE_PATH}/rename_new"
old_df.write.mode("overwrite").parquet(old_path)
new_df.write.mode("overwrite").parquet(new_path)

# Read both — columns don't align!
print("Problem: old file has 'customer_id', new file has 'cust_id'")
combined_broken = spark.read.option("mergeSchema", True).parquet(old_path, new_path)
print(f"Combined schema: {combined_broken.columns}")
combined_broken.show(5)
print("Both columns present — old rows have null cust_id, new rows have null customer_id!")
print()

# Safe rename: keep old name as alias, migrate data
print("Safe migration pattern:")
print("  1. Add new column name (keep old too)")
print("  2. Backfill new column from old")
print("  3. Once all files updated: drop old column")
migration_df = old_df.withColumn("cust_id", F.col("customer_id"))
print(f"Migration schema: {migration_df.columns}")

## Summary: Schema Evolution Rules

| Change | Safe | How |
|---|---|---|
| Add column | ✅ Yes | Old files return null for new column |
| Drop column | ✅ Yes | New files don't write it; old files still have it |
| Widen type (int→long) | ✅ Yes | Read old files with wider schema |
| Narrow type (long→int) | ❌ No | Risk of overflow/truncation |
| Rename column | ⚠️ Care | Add+migrate+drop pattern |

### mergeSchema quick reference
```python
# Read multiple versions with different schemas
spark.read.option("mergeSchema", True).parquet(v1_path, v2_path, v3_path)

# Or from one directory containing mixed-version files
spark.read.option("mergeSchema", True).parquet("/data/table/")
```

**Rule:** Use Iceberg or Delta for production tables with frequent schema changes
— they have first-class schema evolution support with full history.
